# NAMAA Kaggle V6: researched compatibility fix
Isolated `virtualenv` with cleaned PYTHONPATH, matched official PyTorch 2.6.0 + torchaudio 2.6.0 CUDA12.6, fresh interpreter. Logs unhidden pip errors, checks actual audio output. No server usage.

In [ ]:
import os,sys,subprocess
from pathlib import Path
env=os.environ.copy()
env.pop("PYTHONPATH",None)
env.pop("PYTHONHOME",None)
env["PYTHONNOUSERSITE"]="1"
env["PIP_NO_CACHE_DIR"]="1"
env["PIP_PROGRESS_BAR"]="off"
env["HF_HOME"]="/kaggle/working/namaa_hf_cache"
base="/kaggle/working/namaa_venv_v6"
print("PHASE_1_BOOTSTRAP_VIRTUALENV",flush=True)
subprocess.run([sys.executable,"-m","pip","install","-q","virtualenv"],check=True,env=env)
subprocess.run([sys.executable,"-m","virtualenv","--clear","--no-download",base],check=True,env=env)
py=base+"/bin/python"
print("PHASE_2_CHECK_PIP",flush=True)
subprocess.run([py,"-m","pip","--version"],check=True,env=env,timeout=60)
print("PHASE_3_INSTALL_TORCH_PAIR_2_6_CU126",flush=True)
subprocess.run([py,"-m","pip","install","torch==2.6.0","torchaudio==2.6.0","--index-url","https://download.pytorch.org/whl/cu126"],check=True,env=env,timeout=1200)
print("PHASE_4_INSTALL_CHATTERBOX",flush=True)
subprocess.run([py,"-m","pip","install","chatterbox-tts==0.1.7","huggingface_hub"],check=True,env=env,timeout=900)
print("PHASE_5_PRECHECK_FRESH_PYTHON",flush=True)
subprocess.run([py,"-c","import torch,torchaudio;from chatterbox.mtl_tts import ChatterboxMultilingualTTS;print('PRECHECK_OK',torch.__version__,torchaudio.__version__,torch.cuda.is_available(),flush=True); assert torch.cuda.is_available()"],check=True,env=env,timeout=180)
script=Path("/kaggle/working/namaa_infer_v6.py")
script.write_text("import sys,time\nfrom pathlib import Path\nimport torch,torchaudio\nfrom huggingface_hub import snapshot_download\nfrom safetensors.torch import load_file\nfrom chatterbox.mtl_tts import ChatterboxMultilingualTTS\nprint(\"VERSIONS\",sys.version.split()[0],torch.__version__,torchaudio.__version__,\"GPU\",torch.cuda.is_available(),flush=True)\nassert torch.__version__.split(\"+\")[0]==\"2.6.0\"\nassert torchaudio.__version__.split(\"+\")[0]==\"2.6.0\"\nassert torch.cuda.is_available()\nstart=time.monotonic()\nprint(\"BASE_WEIGHT_DOWNLOAD_OR_CACHE\",flush=True)\nmodel=ChatterboxMultilingualTTS.from_pretrained(device=\"cuda\")\nprint(\"NAMAA_WEIGHT_DOWNLOAD_OR_CACHE\",flush=True)\nfolder=snapshot_download(\"NAMAA-Space/NAMAA-Egyptian-TTS\",allow_patterns=[\"t3_mtl23ls_v2.safetensors\"])\nmodel.t3.load_state_dict(load_file(str(Path(folder)/\"t3_mtl23ls_v2.safetensors\"),device=\"cpu\"),strict=True)\nmodel.t3.to(\"cuda\").eval()\nprint(\"NAMAA_READY\",flush=True)\nbegin=time.monotonic()\nwav=model.generate(\"إزيكوا يا جماعة. النهارده هنحكي قصة مانسا موسى، أغنى الملوك في التاريخ.\",language_id=\"ar\")\nout=Path(\"/kaggle/working/namaa_test.wav\")\ntorchaudio.save(str(out),wav.detach().cpu(),model.sr)\nprint(\"SUCCESS\",str(out),\"BYTES\",out.stat().st_size,\"AUDIO_SECONDS\",round(wav.shape[-1]/model.sr,2),\"GENERATE_SECONDS\",round(time.monotonic()-begin,1),\"TOTAL_SECONDS\",round(time.monotonic()-start,1),flush=True)\n",encoding="utf-8")
print("PHASE_6_INFERENCE",flush=True)
subprocess.run([py,"-u",str(script)],check=True,env=env,timeout=1200)
